Chapter 9: Recommendation Systems

Context:
Online stores, music apps and streaming sites suggest products, songs and movies to their users. Good recommendations help customers find things they like and help businesses sell more.

Problem:
Decide which items to recommend to a customer, using only the purchase (or listening) histories of all customers.

Solution:
Store the histories in an interaction matrix (rows = items, columns = users, 1 = purchased). Start with popularity-based recommendations (most purchased items first). Then use cosine similarity to measure how alike two purchase histories are, and build item-based collaborative filtering (recommend items with similar histories) and user-based collaborative filtering (recommend what the most similar user bought). Finally, apply both to real Last.fm music data.

Tools:
pandas, NumPy (dot product, vector norm, cosine similarity)

Implementation:

In [1]:
import pandas as pd
import numpy as np
interaction=pd.read_csv('https://bradfordtuckfield.com/purchasehistory1.csv')
interaction.set_index("Unnamed: 0", inplace = True)
print(interaction)


            user1  user2  user3  user4  user5
Unnamed: 0                                   
item1           1      1      0      1      1
item2           1      0      1      1      0
item3           1      1      0      1      1
item4           1      0      1      0      1
item5           1      1      0      0      1


Popularity-based recommendations: recommend the items that were purchased the most.

In [2]:
interaction_withcounts=interaction.copy()
interaction_withcounts.loc[:,'counts']=interaction_withcounts.sum(axis=1)
interaction_withcounts=interaction_withcounts.sort_values(by='counts',ascending=False)
print(list(interaction_withcounts.index))

def popularity_based(interaction):
  interaction_withcounts=interaction.copy()
  interaction_withcounts.loc[:,'counts']=interaction_withcounts.sum(axis=1)
  sorted = interaction_withcounts.sort_values(by='counts',ascending=False)
  most_popular=list(sorted.index)
  return(most_popular)

print(popularity_based(interaction))

['item1', 'item3', 'item2', 'item4', 'item5']
['item1', 'item3', 'item2', 'item4', 'item5']


Popularity ignores what each person likes. Instead, treat each item's purchase history as a vector and measure how similar two vectors are with cosine similarity (1 = identical, 0 = nothing in common).

In [3]:
print(list(interaction.loc['item1',:]))

def dot_product(vector1,vector2):
  vector1=list(vector1)
  vector2=list(vector2)
  thedotproduct=np.sum([vector1[k]*vector2[k] for k in range(0,len(vector1))])
  return(thedotproduct)

def vector_norm(vector):
  thenorm=np.sqrt(dot_product(vector,vector))
  return(thenorm)

def cosine_similarity(vector1,vector2):
  thedotproduct=dot_product(vector1,vector2)
  thecosine=thedotproduct/(vector_norm(vector1)*vector_norm(vector2))
  thecosine=np.round(thecosine,4)
  return(thecosine)

[1, 1, 0, 1, 1]


In [4]:
import numpy as np
item1=interaction.loc['item1',:]
item3=interaction.loc['item3',:]
print(cosine_similarity(item1,item3))

item2=list(interaction.loc['item2',:])
item5=list(interaction.loc['item5',:])
print(cosine_similarity(item2,item5))

1.0
0.3333


Item-based collaborative filtering: for someone interested in item1, recommend the other items whose histories are most similar to item1's.

In [5]:
ouritem='item1'
otherrows=[rowname for rowname in interaction.index if rowname!=ouritem]
otheritems=interaction.loc[otherrows,:]
theitem=interaction.loc[ouritem,:]

similarities=[]
for items in otheritems.index:
  similarities.append(cosine_similarity(theitem,otheritems.loc[items,:]))
otheritems['similarities']=similarities
recommendations = list(otheritems.sort_values(by='similarities',ascending=False).index)

print(recommendations)

['item3', 'item5', 'item2', 'item4']


In [6]:
def get_item_recommendations(interaction,itemname):
  otherrows=[rowname for rowname in interaction.index if rowname!=itemname]
  otheritems=interaction.loc[otherrows,:]
  theitem=list(interaction.loc[itemname,:])
  similarities=[]
  for items in otheritems.index:
    similarities.append(cosine_similarity(theitem,list(otheritems.loc[items,:])))
  otheritems['similarities']=similarities
  return list(otheritems.sort_values(by='similarities',ascending=False).index)

print(get_item_recommendations(interaction,'item1'))

['item3', 'item5', 'item2', 'item4']


User-based collaborative filtering: find the user whose history is most similar to ours, and recommend what they bought that we haven't.

In [7]:
user2=interaction.loc[:,'user2']
user5=interaction.loc[:,'user5']
print(cosine_similarity(user2,user5))

user3=interaction.loc[:,'user3']
user5=interaction.loc[:,'user5']
print(cosine_similarity(user3,user5))

0.866
0.3536


In [8]:
def get_similar_users(interaction,username):
  othercolumns=[columnname for columnname in interaction.columns if columnname!=username]
  otherusers=interaction[othercolumns]
  theuser=list(interaction[username])
  similarities=[]
  for users in otherusers.columns:
    similarities.append(cosine_similarity(theuser,list(otherusers.loc[:,users])))
  otherusers.loc['similarities',:]=similarities
  return list(otherusers.sort_values(by='similarities',axis=1,ascending=False).columns)

def get_user_recommendations(interaction,username):
  similar_users=get_similar_users(interaction,username)
  purchase_history=interaction[similar_users[0]]
  purchased=list(purchase_history.loc[purchase_history==1].index)
  purchased2=list(interaction.loc[interaction[username]==1,:].index)
  recs=sorted(list(set(purchased) - set(purchased2)))
  return(recs)

print(get_user_recommendations (interaction,'user2'))

['item4']


/tmp/ipykernel_5874/3494024975.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  otherusers.loc['similarities',:]=similarities


Case study: real music-listening data from Last.fm (users in Germany).

In [9]:
import pandas as pd
lastfm = pd.read_csv("https://bradfordtuckfield.com/lastfm-matrix-germany.csv")
print(lastfm.head())

lastfm.drop(['user'],axis=1,inplace=True)

lastfmt=lastfm.T
print(lastfmt.shape)

   user  a perfect circle  abba  ac/dc  adam green  aerosmith  afi  air  \
0     1                 0     0      0           0          0    0    0   
1    33                 0     0      0           1          0    0    0   
2    42                 0     0      0           0          0    0    0   
3    51                 0     0      0           0          0    0    0   
4    62                 0     0      0           0          0    0    0   

   alanis morissette  alexisonfire  ...  timbaland  tom waits  tool  \
0                  0             0  ...          0          0     0   
1                  0             0  ...          0          0     0   
2                  0             0  ...          0          0     0   
3                  0             0  ...          0          0     0   
4                  0             0  ...          0          0     0   

   tori amos  travis  trivium  u2  underoath  volbeat  yann tiersen  
0          0       0        0   0          0        

In [10]:
print(get_item_recommendations(lastfmt,'abba')[0:10])
print(get_user_recommendations(lastfmt,0)[0:3])

['madonna', 'robbie williams', 'elvis presley', 'michael jackson', 'queen', 'the beatles', 'kelly clarkson', 'groove coverage', 'duffy', 'mika']


/tmp/ipykernel_5874/2786876054.py:15: RuntimeWarning: invalid value encountered in scalar divide
  thecosine=thedotproduct/(vector_norm(vector1)*vector_norm(vector2))
/tmp/ipykernel_5874/3494024975.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  otherusers.loc['similarities',:]=similarities


['billy talent', 'bob marley', 'die toten hosen']


Conclusion:
Recommendation systems suggest items using purchase histories. Popularity-based recommendations are simple but give everyone the same list. Cosine similarity measures how alike two histories are: item1 and item3 scored 1.0 (identical), while item2 and item5 scored 0.3333 (different). Item-based collaborative filtering recommended item3, item5, item2, item4 for item1. User-based collaborative filtering found that user5 is most similar to user2 (0.866) and recommended item4. On real Last.fm data, the same functions recommended Madonna and Robbie Williams to ABBA fans. More advanced approaches include content-based filtering, which uses information about the items themselves.